# 🦙 LLaMA Factory — Complete Fine-Tuning Guide
> **LLaMA Factory** is an all-in-one fine-tuning framework for LLMs that wraps `transformers`, `peft`, `bitsandbytes`, and `trl` into a simple UI + CLI interface — enabling SFT, LoRA, QLoRA, DPO, RLHF, and Full Fine-Tuning with zero boilerplate code.

---

## 📋 What You'll Learn in This Notebook
| Section | Topics Covered |
|---|---|
| **1. Setup** | Clone, install, WebUI launch |
| **2. Dataset Formats** | Alpaca, ShareGPT, DPO formats |
| **3. WebUI Workflow** | Step-by-step GUI fine-tuning |
| **4. CLI Training** | YAML configs, CLI commands |
| **5. QLoRA Training** | Gemma + unix-commands dataset |
| **6. Inference** | Adapter loading, merged model loading |
| **7. Export & Merge** | Merge adapters → single model file |
| **8. API Server** | OpenAI-compatible endpoint |
| **9. Advanced Config** | RLHF, DPO, GaLore, Unsloth |

---

## 🧠 LLaMA Factory Architecture

```
┌──────────────────────────────────────────────────────────────────────────────┐
│                         LLAMA FACTORY ARCHITECTURE                            │
├──────────────────────┬───────────────────────────────────────────────────────┤
│  UI Layer            │  CLI Layer                                              │
│  (LlamaBoard/WebUI)  │  (llamafactory-cli)                                    │
│  - Browser-based     │  - YAML configs                                         │
│  - Zero code needed  │  - Scriptable, automatable                              │
├──────────────────────┴───────────────────────────────────────────────────────┤
│                        LLaMA Factory Core Engine                               │
│   SFT   │   LoRA/QLoRA  │   DPO   │   PPO/RLHF   │   Reward Model   │  KTO  │
├─────────────────────────────────────────────────────────────────────────────-─┤
│                     Underlying Libraries                                        │
│  Hugging Face Transformers  │  PEFT  │  BitsAndBytes  │  TRL  │  Accelerate  │
└──────────────────────────────────────────────────────────────────────────────┘
```

---
## 1. 🔧 Environment Setup

LLaMA Factory runs best in **Google Colab** (T4/A100 free tier) or any Linux machine with CUDA GPU.

### Installation Steps
```
1. Clone the LLaMA Factory repository from GitHub
2. Navigate into the project directory
3. Install Python dependencies (requirements.txt)
4. Install in editable mode (pip install -e .)
5. Authenticate with Hugging Face (to download gated models)
6. Launch WebUI or use CLI
```

In [ ]:
# ── Step 1.1: Clone LLaMA Factory repository ─────────────────────────────
# This downloads the full LLaMA Factory source code from GitHub
!git clone https://github.com/hiyouga/LLaMA-Factory.git
!ls LLaMA-Factory  # Verify the clone was successful

In [ ]:
# ── Step 1.2: Navigate into LLaMA-Factory directory ─────────────────────
%cd LLaMA-Factory

# Verify we are in the correct directory
!pwd
!ls

In [ ]:
# ── Step 1.3: Install all required dependencies ──────────────────────────
# requirements.txt contains: transformers, peft, bitsandbytes, trl, accelerate, gradio, etc.
!pip install -r requirements.txt

# Install bitsandbytes separately (sometimes version mismatch occurs)
!pip install "bitsandbytes>=0.39.0"

# Install LLaMA Factory as editable package (makes CLI tools available)
!pip install -e .

# Verify installation
!llamafactory-cli version

In [ ]:
# ── Step 1.4: Hugging Face Authentication ────────────────────────────────
# Required for gated models (LLaMA-3, Gemma, Mistral)
# Get your token from: https://huggingface.co/settings/tokens
!huggingface-cli login

# Alternative: Set token via environment variable
import os
# os.environ["HF_TOKEN"] = "hf_your_token_here"  # Uncomment and fill in

---
## 2. 📂 Dataset Formats

LLaMA Factory understands **3 standard dataset formats**. The format you choose depends on your training stage.

### Format Comparison

| Format | Use Case | Required Fields |
|---|---|---|
| **Alpaca** | SFT / Instruction tuning | `instruction`, `input`, `output` |
| **ShareGPT** | Multi-turn chat fine-tuning | `conversations` list with `from`/`value` |
| **DPO** | Preference alignment | `prompt`, `chosen`, `rejected` |

---

### 📌 Format 1: Alpaca Format (SFT / Instruction Tuning)
Best for: Teaching the model to follow instructions and answer questions.

```json
[
  {
    "instruction": "Explain the ls command in Linux.",
    "input": "",
    "output": "The ls command lists directory contents. Use -l for long format, -a for hidden files."
  },
  {
    "instruction": "What does chmod 755 do?",
    "input": "Context: Linux file permissions",
    "output": "chmod 755 sets: Owner=rwx (read/write/execute), Group=r-x, Others=r-x"
  }
]
```

### 📌 Format 2: ShareGPT Format (Multi-turn Chat)
Best for: Fine-tuning chatbots with multi-turn conversation history.

```json
[
  {
    "conversations": [
      {"from": "system",    "value": "You are a Linux expert assistant."},
      {"from": "human",     "value": "How do I find all .log files?"},
      {"from": "gpt",       "value": "Use: find / -name '*.log' 2>/dev/null"},
      {"from": "human",     "value": "How do I delete them?"},
      {"from": "gpt",       "value": "Use: find / -name '*.log' -delete"}
    ]
  }
]
```

### 📌 Format 3: DPO Format (Preference Alignment)
Best for: Teaching the model WHICH responses are better (alignment stage).

```json
[
  {
    "prompt": "How do I list files in Linux?",
    "chosen": "Use the 'ls' command. For detailed info, use 'ls -la' to show permissions and sizes.",
    "rejected": "I don't know about Linux commands."
  }
]
```

In [ ]:
# ── Step 2.1: Create Sample Datasets for Each Format ─────────────────────
import json
import os

os.makedirs("data", exist_ok=True)

# ─── Alpaca Format Dataset ────────────────────────────────────────────────────
alpaca_data = [
    {
        "instruction": "Explain the ls command in Linux.",
        "input": "",
        "output": "The ls command lists directory contents. Common options: -l (long format), -a (show hidden), -h (human-readable sizes), -t (sort by time)."
    },
    {
        "instruction": "What does chmod 755 mean?",
        "input": "Context: Linux file permission system",
        "output": "chmod 755 sets permissions: Owner gets rwx (read, write, execute). Group gets r-x (read, execute only). Others get r-x (read, execute only)."
    },
    {
        "instruction": "How do I find a file by name in Linux?",
        "input": "",
        "output": "Use the find command: `find /path -name 'filename.txt'`. For case-insensitive search: `find /path -iname 'filename.txt'`."
    },
    {
        "instruction": "What is a pipe (|) in Linux?",
        "input": "",
        "output": "A pipe (|) passes the output of one command as input to another. Example: `ls -l | grep '.txt'` lists files and filters for .txt files."
    }
]

with open("data/unix_alpaca.json", "w") as f:
    json.dump(alpaca_data, f, indent=2)

print("✅ Alpaca format dataset created: data/unix_alpaca.json")
print(f"   Samples: {len(alpaca_data)}")

# ─── ShareGPT Format Dataset ──────────────────────────────────────────────────
sharegpt_data = [
    {
        "conversations": [
            {"from": "system",  "value": "You are a Linux expert assistant. Provide clear, accurate terminal commands."},
            {"from": "human",   "value": "How do I find all .log files in my system?"},
            {"from": "gpt",     "value": "Use this command: `find / -name '*.log' 2>/dev/null`\nThe 2>/dev/null suppresses permission error messages."},
            {"from": "human",   "value": "How do I count how many .log files there are?"},
            {"from": "gpt",     "value": "Pipe the result to wc: `find / -name '*.log' 2>/dev/null | wc -l`\nwc -l counts lines, and each line = one file."}
        ]
    },
    {
        "conversations": [
            {"from": "human",   "value": "What is the difference between > and >> in Linux?"},
            {"from": "gpt",     "value": "> redirects output to a file, OVERWRITING any existing content.\n>> appends output to a file, KEEPING existing content.\nExample:\n  echo 'hello' > file.txt   # Creates/overwrites file.txt\n  echo 'world' >> file.txt  # Appends to file.txt"}
        ]
    }
]

with open("data/unix_sharegpt.json", "w") as f:
    json.dump(sharegpt_data, f, indent=2)

print("✅ ShareGPT format dataset created: data/unix_sharegpt.json")

# ─── DPO Format Dataset ───────────────────────────────────────────────────────
dpo_data = [
    {
        "prompt": "How do I list all files including hidden ones in Linux?",
        "chosen": "Use `ls -la` to list all files including hidden ones (those starting with .). The -l flag gives long format with permissions, ownership, and size. The -a flag shows all files including hidden ones.",
        "rejected": "Just type ls in the terminal and you should see all your files."
    },
    {
        "prompt": "What command removes a directory in Linux?",
        "chosen": "Use `rm -r directory_name` to remove a directory and all its contents. Use `rmdir directory_name` for empty directories only. Warning: rm -r is irreversible — there is no trash bin.",
        "rejected": "Delete it."
    }
]

with open("data/unix_dpo.json", "w") as f:
    json.dump(dpo_data, f, indent=2)

print("✅ DPO format dataset created: data/unix_dpo.json")
print("\n📁 Created datasets:")
for f in os.listdir("data"):
    size = os.path.getsize(f"data/{f}")
    print(f"   {f} — {size} bytes")

In [ ]:
# ── Step 2.2: Register Custom Dataset with LLaMA Factory ─────────────────
# LLaMA Factory reads dataset_info.json to know about available datasets
# We add our custom dataset entry to this file

import json

dataset_info_path = "data/dataset_info.json"

# Load existing dataset_info.json (LLaMA Factory ships with many built-in datasets)
try:
    with open(dataset_info_path) as f:
        dataset_info = json.load(f)
except FileNotFoundError:
    dataset_info = {}

# Register our custom Alpaca dataset
dataset_info["unix_commands_alpaca"] = {
    "file_name": "unix_alpaca.json",
    "formatting": "alpaca",
    "columns": {
        "prompt": "instruction",
        "query": "input",
        "response": "output"
    }
}

# Register ShareGPT dataset
dataset_info["unix_commands_sharegpt"] = {
    "file_name": "unix_sharegpt.json",
    "formatting": "sharegpt",
    "columns": {
        "messages": "conversations"
    },
    "tags": {
        "role_tag": "from",
        "content_tag": "value",
        "user_tag": "human",
        "assistant_tag": "gpt",
        "system_tag": "system"
    }
}

# Save updated dataset_info.json
with open(dataset_info_path, "w") as f:
    json.dump(dataset_info, f, indent=2)

print("✅ Custom datasets registered in dataset_info.json")
print(f"   Total datasets available: {len(dataset_info)}")

---
## 3. 🖥️ WebUI (LlamaBoard) — Step-by-Step Guide

The WebUI provides a full GUI for fine-tuning without writing code.

### WebUI Sections

```
┌──────────────────────────────────────────────────────────────────┐
│                     LlamaBoard WebUI Layout                       │
├─────────────┬──────────────┬────────────┬────────────────────────┤
│   Train     │   Evaluate   │    Chat    │        Export          │
├─────────────┴──────────────┴────────────┴────────────────────────┤
│                    Configuration Panel                             │
│  ┌────────────┐  ┌──────────────┐  ┌──────────────────────────┐  │
│  │ Model      │  │  Dataset     │  │    Training Params        │  │
│  │ Selection  │  │  & Format    │  │    (LR, Epochs, LoRA)    │  │
│  └────────────┘  └──────────────┘  └──────────────────────────┘  │
├──────────────────────────────────────────────────────────────────┤
│             Start Training / Loss Curves / Chat Test             │
└──────────────────────────────────────────────────────────────────┘
```

### 8-Step WebUI Workflow

| Step | Action | What to Set |
|---|---|---|
| **1** | Select Base Model | Model name from HuggingFace (e.g. `google/gemma-1.1-2b-it`) |
| **2** | Choose Fine-tuning Type | LoRA / Full / Freeze |
| **3** | Set Quantization | None / 8-bit / 4-bit (QLoRA) |
| **4** | Load Dataset | Select from dropdown or upload file |
| **5** | Set Training Params | Epochs, LR, Batch Size, LoRA rank |
| **6** | Start Training | Click "Start" — watch loss curves live |
| **7** | Evaluate / Chat | Test model quality in the Chat tab |
| **8** | Export | Merge LoRA + base → single model file |

In [ ]:
# ── Step 3.1: Launch LlamaBoard WebUI ────────────────────────────────────
# The WebUI runs a Gradio server accessible via a public URL (in Colab)
import os
os.environ["GRADIO_SHARE"] = "1"     # Enable public sharing link in Colab

# Method 1: Using the installed CLI command
# !llamafactory-cli webui

# Method 2: Using Python module
from llamafactory.webui.interface import create_ui

ui = create_ui()
ui.launch(
    share=True,         # Creates public ngrok URL accessible from anywhere
    server_port=7860,   # Default Gradio port
    quiet=True          # Suppress verbose output
)

---
## 4. ⚙️ CLI Training — YAML Configuration System

LLaMA Factory CLI uses **YAML configuration files** to define every aspect of training.
This makes training reproducible, scriptable, and easy to version-control.

### Why YAML?
- All parameters in one file — no long command-line argument chains
- Easy to share and reproduce experiments
- Version-controllable with git

### YAML Config Sections
```
model_name_or_path    ← Which pretrained model to use
template              ← Chat format template (llama3, gemma, qwen, etc.)
stage                 ← Training type (sft, dpo, ppo, rm, pt)
finetuning_type       ← lora / full / freeze
dataset settings      ← What data to train on
training settings     ← Epochs, LR, batch size
LoRA settings         ← r, alpha, dropout, target modules
quantization settings ← 4-bit, 8-bit, none
output settings       ← Where to save checkpoints
```

### Chat Templates Reference
| Model Family | Template Name |
|---|---|
| LLaMA-2 | `llama2` |
| LLaMA-3 / LLaMA-3.1 | `llama3` |
| Gemma / Gemma-2 | `gemma` |
| Mistral | `mistral` |
| Qwen / Qwen-2 | `qwen` |
| Phi-3 | `phi` |
| ChatGLM-3 | `chatglm3` |
| DeepSeek | `deepseek` |

In [ ]:
# ── Step 4.1: Create YAML Configuration Files ───────────────────────────

# ─── Config 1: Gemma-2B QLoRA SFT (4-bit) ────────────────────────────────────
gemma_qlora_sft_yaml = """# ============================================================
# LLaMA Factory — QLoRA SFT Config for Gemma-2B
# Fine-tunes Gemma on Unix commands instruction dataset
# ============================================================

# ── Model Configuration ─────────────────────────────────────
model_name_or_path: google/gemma-1.1-2b-it  # Base model from Hugging Face Hub
template: gemma                               # Chat template for Gemma models

# ── Training Stage ──────────────────────────────────────────
stage: sft                 # Supervised Fine-Tuning (instruction following)
do_train: true             # Enable training mode

# ── Fine-tuning Method ──────────────────────────────────────
finetuning_type: lora      # LoRA: train only small adapter matrices

# ── Dataset Configuration ───────────────────────────────────
dataset: unix_commands_alpaca   # Dataset name (registered in dataset_info.json)
dataset_dir: data               # Directory containing dataset files
cutoff_len: 1024                # Max sequence length (truncate longer sequences)
val_size: 0.1                   # Use 10% of data for validation

# ── Quantization (QLoRA) ────────────────────────────────────
quantization_bit: 4             # Load base model in 4-bit (huge VRAM savings)
quantization_method: bitsandbytes  # Use bitsandbytes for quantization

# ── LoRA Configuration ──────────────────────────────────────
lora_rank: 16               # r=16: adapter matrix rank (higher = more capacity)
lora_alpha: 32              # alpha = 2*r (standard scaling convention)
lora_dropout: 0.05          # Regularization dropout on LoRA activations
lora_target: all            # Apply LoRA to all linear layers

# ── Training Hyperparameters ────────────────────────────────
output_dir: output/gemma-unix-qlora   # Where to save checkpoints and adapters
overwrite_output_dir: true
num_train_epochs: 3.0               # Train for 3 complete dataset passes
per_device_train_batch_size: 1      # 1 sample per GPU (small GPU friendly)
gradient_accumulation_steps: 8      # Effective batch = 1 * 8 = 8
learning_rate: 0.0002               # 2e-4 (higher LR for LoRA, fewer params)
lr_scheduler_type: cosine           # Cosine decay (smooth, stable)
warmup_ratio: 0.03                  # Warm up LR for first 3% of training

# ── Precision & Speed ───────────────────────────────────────
bf16: true                          # BF16 mixed precision (better than FP16 for training)
gradient_checkpointing: true        # Recompute activations to save VRAM

# ── Logging & Saving ────────────────────────────────────────
logging_steps: 10                   # Log loss/metrics every 10 steps
save_steps: 100                     # Save checkpoint every 100 steps
save_total_limit: 2                 # Keep only last 2 checkpoints
report_to: none                     # Disable W&B / TensorBoard
"""

with open("train_gemma_qlora.yaml", "w") as f:
    f.write(gemma_qlora_sft_yaml)

print("✅ Created: train_gemma_qlora.yaml")

# ─── Config 2: LLaMA-3 8B QLoRA SFT ─────────────────────────────────────────
llama3_qlora_yaml = """# ============================================================
# LLaMA Factory — QLoRA SFT Config for LLaMA-3 8B
# ============================================================
model_name_or_path: meta-llama/Meta-Llama-3-8B-Instruct
template: llama3
stage: sft
do_train: true
finetuning_type: lora
dataset: unix_commands_alpaca
dataset_dir: data
cutoff_len: 1024
val_size: 0.1
quantization_bit: 4
quantization_method: bitsandbytes
lora_rank: 64          # Higher rank for larger model (more capacity)
lora_alpha: 16
lora_dropout: 0.05
lora_target: all
output_dir: output/llama3-unix-qlora
num_train_epochs: 3.0
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 0.0002
lr_scheduler_type: cosine
warmup_ratio: 0.03
bf16: true
gradient_checkpointing: true
logging_steps: 10
save_steps: 100
save_total_limit: 2
report_to: none
"""

with open("train_llama3_qlora.yaml", "w") as f:
    f.write(llama3_qlora_yaml)

print("✅ Created: train_llama3_qlora.yaml")

# ─── Config 3: DPO Alignment Config ──────────────────────────────────────────
dpo_yaml = """# ============================================================
# LLaMA Factory — DPO Alignment Config
# Run AFTER SFT to improve response quality/preference alignment
# ============================================================
model_name_or_path: output/gemma-unix-qlora  # Load SFT-trained model
template: gemma
stage: dpo              # Direct Preference Optimization stage
do_train: true
finetuning_type: lora
dataset: unix_commands_dpo
dataset_dir: data
cutoff_len: 1024
pref_beta: 0.1          # Beta controls how strictly we stay near reference model
pref_loss: sigmoid      # DPO loss type: sigmoid (standard), margin, hinge
output_dir: output/gemma-unix-dpo
num_train_epochs: 1.0   # DPO usually needs fewer epochs than SFT
per_device_train_batch_size: 1
gradient_accumulation_steps: 4
learning_rate: 0.000005  # 5e-6: Much smaller LR for alignment (don't overfit)
bf16: true
gradient_checkpointing: true
logging_steps: 5
save_total_limit: 1
report_to: none
"""

with open("train_dpo.yaml", "w") as f:
    f.write(dpo_yaml)

print("✅ Created: train_dpo.yaml")

# ─── Config 4: Export / Merge Config ─────────────────────────────────────────
export_yaml = """# ============================================================
# LLaMA Factory — Export Config (Merge LoRA → Full Model)
# ============================================================
model_name_or_path: google/gemma-1.1-2b-it  # Original base model
adapter_name_or_path: output/gemma-unix-qlora  # Trained LoRA adapter path
template: gemma
finetuning_type: lora
export_dir: output/gemma-unix-merged          # Where to save merged model
export_size: 2                                # Split into 2GB shards (None = single file)
export_device: cpu                            # Use CPU for merging (saves GPU VRAM)
export_legacy_format: false                   # Use safetensors format
"""

with open("export_gemma.yaml", "w") as f:
    f.write(export_yaml)

print("✅ Created: export_gemma.yaml")
print("\n📁 All YAML configs created:")
for f in ["train_gemma_qlora.yaml", "train_llama3_qlora.yaml", "train_dpo.yaml", "export_gemma.yaml"]:
    print(f"   {f}")

---
## 5. 🚀 Running Training

### CLI Commands Reference

| Command | Purpose |
|---|---|
| `llamafactory-cli train config.yaml` | Start training session |
| `llamafactory-cli chat config.yaml` | CLI interactive chat |
| `llamafactory-cli eval config.yaml` | Evaluate perplexity/metrics |
| `llamafactory-cli export config.yaml` | Merge LoRA + export |
| `llamafactory-cli api config.yaml` | Start OpenAI-compatible API |
| `llamafactory-cli webui` | Launch graphical interface |

### Understanding Training Output

```
{'loss': 2.45, 'learning_rate': 0.0002, 'epoch': 0.15}  ← Early training
{'loss': 1.87, 'learning_rate': 0.00018, 'epoch': 0.58}  ← Getting better
{'loss': 1.21, 'learning_rate': 0.00010, 'epoch': 1.52}  ← Good progress
{'loss': 0.84, 'learning_rate': 0.00003, 'epoch': 2.87}  ← Near convergence
```

**Loss is printed every `logging_steps` steps — this is NOT extra epochs!**
A "step" = one forward+backward pass over one mini-batch.

In [ ]:
# ── Step 5.1: Run QLoRA Training ─────────────────────────────────────────
import os
import subprocess

# Verify YAML config exists before running
config_file = "train_gemma_qlora.yaml"
assert os.path.exists(config_file), f"Config file not found: {config_file}"

# Check GPU availability
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

# Set environment variables
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # Helps with CUDA error debugging
os.environ["WANDB_DISABLED"] = "true"      # Disable Weights & Biases logging

# Run training
print(f"🚀 Starting training with config: {config_file}")
print("=" * 60)
!llamafactory-cli train {config_file}

In [ ]:
# ── Step 5.2: Monitor Training Progress & Checkpoints ────────────────────
import os
import json

output_dir = "output/gemma-unix-qlora"

# Check what was saved
if os.path.exists(output_dir):
    print(f"📁 Training output directory: {output_dir}")
    for item in sorted(os.listdir(output_dir)):
        item_path = os.path.join(output_dir, item)
        if os.path.isdir(item_path):
            files = os.listdir(item_path)
            print(f"   📂 {item}/ ({len(files)} files)")
        else:
            size_mb = os.path.getsize(item_path) / (1024**2)
            print(f"   📄 {item} ({size_mb:.2f} MB)")

    # Read training state for metrics
    trainer_state_path = os.path.join(output_dir, "trainer_state.json")
    if os.path.exists(trainer_state_path):
        with open(trainer_state_path) as f:
            state = json.load(f)
        
        history = state.get("log_history", [])
        print(f"\n📊 Training History ({len(history)} log entries):")
        print(f"   First loss: {history[0].get('loss', 'N/A') if history else 'N/A'}")
        print(f"   Last loss:  {history[-1].get('loss', 'N/A') if history else 'N/A'}")
        print(f"   Best metric: {state.get('best_metric', 'N/A')}")
else:
    print(f"⚠️  Output directory not found: {output_dir}")
    print("   Run the training cell first!")

---
## 6. 🔍 Inference — Loading & Testing Your Fine-tuned Model

After training, you can load the model in **two ways**:

### Method A: Load Base Model + LoRA Adapter (Separate)
- Keep the base model untouched
- Load LoRA adapter on top at runtime
- ✅ Flexible — easy to swap adapters
- ⚠️ Slightly slower (two model loads)

### Method B: Load Merged Model (Adapter fused into base)
- Base weights + LoRA weights permanently combined
- ✅ Fastest inference — single model load
- ✅ Compatible with all serving frameworks (vLLM, Ollama, llama.cpp)
- ⚠️ Larger file size (full model)

In [ ]:
# ── Step 6.1: Method A — Inference with Base + LoRA Adapter ─────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# Configuration
BASE_MODEL = "google/gemma-1.1-2b-it"
ADAPTER_PATH = "output/gemma-unix-qlora"   # Your trained LoRA folder

# Load tokenizer from base model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# Load base model (in original precision)
print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,        # FP16 for efficient inference
    device_map="auto"                  # Auto-assign to GPU
)

# Apply trained LoRA adapter on top of base model
print("Applying LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

model.eval()  # Disable dropout for inference
print("✅ Model ready!")

# Test inference
def generate_response(prompt: str, max_new_tokens: int = 200) -> str:
    """
    Generate a response from the fine-tuned model given a prompt.
    Uses Gemma's instruction template format.
    """
    # Format using Gemma's chat template
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    # Tokenize and move to device
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode and clean up
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only the new generated text (after the prompt)
    response = full_text[len(prompt):].strip()
    return response

# Test the model on Unix commands
test_questions = [
    "Can you tell me what ls -l would display?",
    "How do I create a new directory in Linux?",
    "What does grep do and give me an example?"
]

print("\n" + "="*60)
print("FINE-TUNED MODEL RESPONSES")
print("="*60)

for question in test_questions:
    print(f"\nQ: {question}")
    response = generate_response(question)
    print(f"A: {response}")
    print("-"*60)

In [ ]:
# ── Step 6.2: Compare Base Model vs Fine-tuned Model ─────────────────────
# This cell demonstrates the improvement from fine-tuning

test_prompt = "What is the command to show disk usage in Linux?"

print("=" * 65)
print("        COMPARISON: BASE vs FINE-TUNED MODEL")
print("=" * 65)

print(f"\nPrompt: {test_prompt}")
print("-" * 65)

# Fine-tuned model response
print("\n🦙 Fine-tuned Model Response (trained on unix-commands):")
ft_response = generate_response(test_prompt, max_new_tokens=100)
print(ft_response)

# Note: For base model comparison, you would load:
# base_only = AutoModelForCausalLM.from_pretrained(BASE_MODEL)
# The base model would give a generic/less specific answer about disk usage
# vs our fine-tuned model which knows specific Linux commands precisely

print("\n" + "="*65)
print("Key differences after fine-tuning on unix-commands dataset:")
print("  ✅ More precise command syntax (e.g., du -sh, df -h)")
print("  ✅ Correct flag explanations")
print("  ✅ Domain-specific examples")
print("  ✅ Concise, structured responses")

---
## 7. 📦 Export & Merge — Creating a Production-Ready Model

After training, **export merges** the LoRA adapter weights permanently into the base model weights:

$$W_{merged} = W_{base} + \frac{\alpha}{r}(B \times A)$$

The result is a standard model file with no PEFT overhead — compatible with vLLM, Ollama, llama.cpp, TGI.

### Export Methods Comparison

| Method | How | When to Use |
|---|---|---|
| **CLI Export** | `llamafactory-cli export config.yaml` | Production deployment |
| **Python Merge** | `model.merge_and_unload()` | Programmatic control |
| **WebUI Export** | Export tab in LlamaBoard | Beginners |

In [ ]:
# ── Step 7.1: Export via CLI (Merge LoRA → Full Model) ──────────────────
# This reads export_gemma.yaml which we created in Step 4.1
!llamafactory-cli export export_gemma.yaml

# After export, verify the output
import os
export_dir = "output/gemma-unix-merged"
if os.path.exists(export_dir):
    print(f"\n✅ Export successful!")
    print(f"📁 Merged model saved to: {export_dir}")
    for f in sorted(os.listdir(export_dir)):
        size_mb = os.path.getsize(os.path.join(export_dir, f)) / (1024**2)
        print(f"   {f} — {size_mb:.2f} MB")
else:
    print("⚠️ Export directory not found. Run training first.")

In [ ]:
# ── Step 7.2: Python-based Merge (Alternative to CLI) ───────────────────
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_MODEL = "google/gemma-1.1-2b-it"
ADAPTER_PATH = "output/gemma-unix-qlora"
MERGED_OUTPUT = "output/gemma-unix-merged-python"

print("Step 1: Loading base model...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

print("Step 2: Loading and applying LoRA adapter...")
model_with_adapter = PeftModel.from_pretrained(base, ADAPTER_PATH)

print("Step 3: Merging LoRA weights into base (merge_and_unload)...")
# merge_and_unload() permanently fuses W₀ + (α/r)(BA) into base weights
# Returns a standard AutoModelForCausalLM with no PEFT overhead
merged_model = model_with_adapter.merge_and_unload()

print("Step 4: Saving merged model...")
import os
os.makedirs(MERGED_OUTPUT, exist_ok=True)
merged_model.save_pretrained(MERGED_OUTPUT, safe_serialization=True)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.save_pretrained(MERGED_OUTPUT)

print(f"\n✅ Merged model saved to: {MERGED_OUTPUT}")
print("This model can now be used without PEFT library!")
print("Compatible with: vLLM, Ollama, llama.cpp, TGI, text-generation-webui")

In [ ]:
# ── Step 7.3: Load and Test Merged Model ────────────────────────────────
# Merged model loads like any standard HuggingFace model — no PeftModel needed!

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MERGED_PATH = "output/gemma-unix-merged"

print("Loading merged model (no PEFT required)...")
tokenizer_merged = AutoTokenizer.from_pretrained(MERGED_PATH)
model_merged = AutoModelForCausalLM.from_pretrained(
    MERGED_PATH,
    torch_dtype=torch.float16,
    device_map="auto"
)
model_merged.eval()

# Quick test
prompt = "Explain what the grep command does in Linux."
inputs = tokenizer_merged(prompt, return_tensors="pt").to(model_merged.device)

with torch.no_grad():
    outputs = model_merged.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer_merged.eos_token_id
    )

response = tokenizer_merged.decode(outputs[0], skip_special_tokens=True)
print(f"\nPrompt: {prompt}")
print(f"Response: {response}")

---
## 8. 🌐 OpenAI-Compatible API Server

LLaMA Factory can expose your fine-tuned model as an **OpenAI-compatible REST API**.
This lets you use your model with any tool that supports the OpenAI API (LangChain, OpenWebUI, etc.)

### API Endpoints
| Endpoint | Method | Description |
|---|---|---|
| `/v1/chat/completions` | POST | Chat completion (same as OpenAI) |
| `/v1/completions` | POST | Text completion |
| `/v1/models` | GET | List available models |

In [ ]:
# ── Step 8.1: Configure and Start API Server ─────────────────────────────

api_yaml = """# ============================================================
# LLaMA Factory — API Server Config
# Starts an OpenAI-compatible endpoint for your fine-tuned model
# ============================================================

model_name_or_path: output/gemma-unix-merged  # Path to merged model
template: gemma
finetuning_type: full         # Merged model = no LoRA adapters

# API Server Settings
infer_backend: huggingface    # huggingface / vllm / sglang
"""

with open("api_config.yaml", "w") as f:
    f.write(api_yaml)

print("✅ API config created: api_config.yaml")
print()
print("To start the API server, run:")
print("  llamafactory-cli api api_config.yaml")
print()
print("The server will be available at: http://0.0.0.0:8000")
print()
print("Test with curl:")
print("""  curl http://localhost:8000/v1/chat/completions \\
    -H "Content-Type: application/json" \\
    -d '{
      "model": "gemma-unix",
      "messages": [{"role": "user", "content": "What does ls -la do?"}],
      "max_tokens": 200
    }'""")

In [ ]:
# ── Step 8.2: Python Client for API Server ───────────────────────────────
# Once the API server is running, you can call it with the openai Python library

try:
    from openai import OpenAI

    # Point OpenAI client to your local LLaMA Factory server
    client = OpenAI(
        base_url="http://localhost:8000/v1",
        api_key="llamafactory"  # LLaMA Factory accepts any non-empty API key
    )

    response = client.chat.completions.create(
        model="gemma-unix",  # Model name (any string works)
        messages=[
            {"role": "system", "content": "You are a Linux expert assistant."},
            {"role": "user",   "content": "What does the grep command do?"}
        ],
        max_tokens=200,
        temperature=0.7
    )

    print("API Response:")
    print(response.choices[0].message.content)

except ImportError:
    print("Install openai library: pip install openai")
    print("Then ensure your API server is running: llamafactory-cli api api_config.yaml")

---
## 9. ⚡ Advanced Techniques

### 9.1 Training Speed Enhancers

| Method | VRAM Saving | Speed Boost | Compatibility |
|---|---|---|---|
| **Flash Attention** | ~20-30% | 2-4x faster attention | NVIDIA Ampere+ (A100, 3090+) |
| **Unsloth** | ~50% less VRAM | 2x faster overall | Most NVIDIA GPUs |
| **Liger Kernel** | 20% less | 1.5x faster | NVIDIA GPUs with CUDA |
| **Gradient Checkpointing** | ~60% less | Slight slowdown | All GPUs |

### 9.2 Advanced LoRA Variants

| Variant | Description | When to Use |
|---|---|---|
| **Standard LoRA** | `A × B` decomposition | Default, always works |
| **rsLoRA** | Rank-stabilized scaling | Deeper LoRA ranks (r>32) |
| **DoRA** | Weight-decomposed LoRA | Higher quality, slight overhead |
| **PiSSA** | Principal Singular Value adaptation | Better initialization |

### 9.3 DeepSpeed Integration (Multi-GPU)

```yaml
# Add to your YAML config for multi-GPU training
deepspeed: ds_z3_config.json   # ZeRO Stage 3 (maximum memory savings)
```

| ZeRO Stage | What's Sharded | Best For |
|---|---|---|
| Stage 1 | Optimizer states | 2-4 GPUs |
| Stage 2 | + Gradients | 4-8 GPUs |
| Stage 3 | + Model params | 8+ GPUs (70B+ models) |

### 9.4 RoPE Scaling (Long Context)

| Method | Context Ext. | Quality |
|---|---|---|
| `linear` | 2-4x original | Moderate |
| `dynamic` | 4-8x | Good |
| `yarn` | 8-32x | Best (SOTA) |
| `llama3` | LLaMA-3 specific | Best for LLaMA-3 |

In [ ]:
# ── Step 9.1: Unsloth-Accelerated Training Config ────────────────────────
# Unsloth provides ~2x speedup and ~50% VRAM reduction
# Best for: Single GPU training on Colab T4/A100

unsloth_yaml = """# ============================================================
# LLaMA Factory — Unsloth Accelerated QLoRA
# 2x faster + 50% less VRAM than standard training
# ============================================================

model_name_or_path: google/gemma-1.1-2b-it
template: gemma
stage: sft
do_train: true
finetuning_type: lora

dataset: unix_commands_alpaca
dataset_dir: data
cutoff_len: 2048          # Unsloth handles longer sequences efficiently

quantization_bit: 4
lora_rank: 32
lora_alpha: 64
lora_dropout: 0.0         # Unsloth works best with dropout=0
lora_target: all

# ─── Unsloth Speed Booster ─────────────────────────────────
flash_attn: fa2             # Flash Attention 2 (requires FA2 installed)
use_unsloth: true           # Enable Unsloth kernels

output_dir: output/gemma-unsloth
num_train_epochs: 3.0
per_device_train_batch_size: 2      # Unsloth enables larger batches!
gradient_accumulation_steps: 4      # Effective batch = 2*4 = 8
learning_rate: 0.0002
lr_scheduler_type: cosine
warmup_ratio: 0.03
bf16: true
gradient_checkpointing: true        # Unsloth custom gradient checkpointing
logging_steps: 10
save_total_limit: 1
report_to: none
"""

with open("train_unsloth.yaml", "w") as f:
    f.write(unsloth_yaml)

print("✅ Unsloth config created: train_unsloth.yaml")
print("\nTo run: llamafactory-cli train train_unsloth.yaml")

In [ ]:
# ── Step 9.2: GaLore — Gradient Low-Rank Projection ─────────────────────
# GaLore compresses GRADIENTS (not weights) into low-rank space
# Enables full fine-tuning of large models with LoRA-level VRAM usage

galore_yaml = """# ============================================================
# LLaMA Factory — GaLore Full Fine-tuning
# Full FT with VRAM similar to LoRA via gradient compression
# ============================================================

model_name_or_path: google/gemma-1.1-2b-it
template: gemma
stage: sft
do_train: true
finetuning_type: full       # Full fine-tuning (all weights)

# GaLore compresses the gradients to save VRAM
use_galore: true            # Enable GaLore
galore_rank: 16             # Low-rank dimension for gradient compression
galore_update_interval: 200 # Update projection matrix every 200 steps
galore_scale: 0.25          # Gradient update scaling factor
galore_proj_type: std       # std = standard GaLore projection

dataset: unix_commands_alpaca
dataset_dir: data
output_dir: output/gemma-galore

num_train_epochs: 2.0
per_device_train_batch_size: 1
gradient_accumulation_steps: 8
learning_rate: 0.0001       # Lower LR for full FT
bf16: true
gradient_checkpointing: true
logging_steps: 10
report_to: none
"""

with open("train_galore.yaml", "w") as f:
    f.write(galore_yaml)

print("✅ GaLore config created: train_galore.yaml")

---
## 10. 🏆 Preference Alignment in LLaMA Factory

### Training Pipeline for Full Alignment
```
Stage 1: SFT      → Train on instruction-response pairs
Stage 2: RM       → Train Reward Model to score good vs bad responses
Stage 3: PPO      → Use Reward Model to optimize policy (full RLHF)
    OR
Stage 2: DPO      → Direct Preference Optimization (no reward model needed)
    OR
Stage 2: KTO      → Kahneman-Tversky Optimization (improved DPO variant)
```

### Stage Comparison in LLaMA Factory
| Stage Key | Training Type | Data Required |
|---|---|---|
| `pt` | Pretraining (raw text) | `{"text": "..."}`|
| `sft` | Supervised Fine-Tuning | Alpaca/ShareGPT format |
| `rm` | Reward Model Training | `{prompt, chosen, rejected}` |
| `ppo` | PPO Reinforcement Learning | Requires trained RM |
| `dpo` | Direct Preference Optimization | `{prompt, chosen, rejected}` |
| `kto` | KTO Optimization | `{prompt, response, label: bool}` |

In [ ]:
# ── Step 10.1: Prepare KTO Dataset & Run DPO/KTO Training ──────────────
import json

# KTO format: each example has prompt + response + label (true=good, false=bad)
kto_data = [
    {
        "prompt": "How do I list files in Linux?",
        "response": "Use the 'ls' command. For detailed info with permissions, use 'ls -la'.",
        "label": True    # ← This is a GOOD response (thumbs up)
    },
    {
        "prompt": "How do I list files in Linux?",
        "response": "I cannot help with that.",
        "label": False   # ← This is a BAD response (thumbs down)
    },
    {
        "prompt": "What does chmod do?",
        "response": "chmod changes file permissions. Example: chmod 755 file.sh makes it executable.",
        "label": True
    },
    {
        "prompt": "What does chmod do?",
        "response": "chmod is a Linux thing I think.",
        "label": False
    }
]

with open("data/unix_kto.json", "w") as f:
    json.dump(kto_data, f, indent=2)

print("✅ KTO dataset created: data/unix_kto.json")

# Register KTO dataset
import json
with open("data/dataset_info.json") as f:
    dataset_info = json.load(f)

dataset_info["unix_commands_kto"] = {
    "file_name": "unix_kto.json",
    "formatting": "alpaca",
    "ranking": True,        # Signals this is a preference dataset
    "columns": {
        "prompt": "prompt",
        "response": "response",
        "kto_tag": "label"
    }
}

with open("data/dataset_info.json", "w") as f:
    json.dump(dataset_info, f, indent=2)

print("✅ KTO dataset registered")

# DPO Training YAML (already created) — run it:
print("\nTo run DPO alignment training:")
print("  llamafactory-cli train train_dpo.yaml")

---
## 11. 🔧 Common Errors & Troubleshooting

### Error Reference Table

| Error | Cause | Fix |
|---|---|---|
| `CUDA out of memory` | VRAM exhausted | Reduce batch_size to 1, enable 4-bit, use gradient_checkpointing |
| `403 Forbidden` (HuggingFace) | Model is gated, not authenticated | Run `huggingface-cli login` |
| `Dataset not found` | Not registered in dataset_info.json | Add entry to `data/dataset_info.json` |
| `Template not found` | Wrong template name | Check template name in supported templates list |
| `Training loss = NaN` | LR too high, data issue | Reduce learning_rate, check dataset format |
| `AttributeError: PeftModel` | PEFT version mismatch | Run `pip install -U peft` |
| `KeyError: conversations` | Wrong ShareGPT column mapping | Verify `columns.messages` in dataset_info |
| `Loss not decreasing` | LR too low or dataset too small | Increase LR, add more data |

In [ ]:
# ── Step 11.1: System Diagnostics ───────────────────────────────────────
import subprocess
import torch
import os

print("=" * 65)
print("           SYSTEM DIAGNOSTICS")
print("=" * 65)

# GPU Info
print("\n[GPU]")
!nvidia-smi --query-gpu=name,driver_version,memory.total,memory.used,memory.free --format=csv

# Package Versions
print("\n[Package Versions]")
import transformers, peft, bitsandbytes, datasets
print(f"  transformers : {transformers.__version__}")
print(f"  peft         : {peft.__version__}")
print(f"  bitsandbytes : {bitsandbytes.__version__}")
print(f"  datasets     : {datasets.__version__}")
print(f"  torch        : {torch.__version__}")
print(f"  CUDA         : {torch.version.cuda}")

# PyTorch VRAM stats
if torch.cuda.is_available():
    print("\n[Current VRAM Usage]")
    allocated = torch.cuda.memory_allocated() / 1024**3
    reserved  = torch.cuda.memory_reserved()  / 1024**3
    total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"  Allocated : {allocated:.2f} GB")
    print(f"  Reserved  : {reserved:.2f} GB")
    print(f"  Total     : {total:.2f} GB")
    print(f"  Free      : {total - reserved:.2f} GB")

# Dataset files
print("\n[Dataset Files]")
for f in sorted(os.listdir("data") if os.path.exists("data") else []):
    size = os.path.getsize(f"data/{f}") / 1024
    print(f"  data/{f} — {size:.1f} KB")

# YAML configs
print("\n[YAML Configs]")
for f in sorted(os.listdir(".")):
    if f.endswith(".yaml"):
        print(f"  {f}")

print("\n✅ Diagnostics complete!")

---
## 12. 📋 Quick Reference Cheat Sheet

### Key Parameters Quick Reference

```yaml
# ─── Most Important Parameters ───────────────────────────────────────────────
model_name_or_path: google/gemma-1.1-2b-it   # Any HuggingFace model ID
template: gemma                               # Match model family!
stage: sft                                    # sft / rm / ppo / dpo / kto / pt
finetuning_type: lora                         # lora / full / freeze

# ─── LoRA ────────────────────────────────────────────────────────────────────
lora_rank: 16           # r: 4=tiny, 8=small, 16=standard, 64=large
lora_alpha: 32          # α: always set to 2×r
lora_target: all        # all = every linear layer (best quality)

# ─── Quantization ─────────────────────────────────────────────────────────────
quantization_bit: 4     # 4=QLoRA (minimum VRAM), 8=INT8, none=full precision

# ─── Training ────────────────────────────────────────────────────────────────
num_train_epochs: 3.0
per_device_train_batch_size: 1
gradient_accumulation_steps: 8    # effective_batch = 1*8 = 8
learning_rate: 0.0002             # LoRA: 2e-4, Full FT: 2e-5
lr_scheduler_type: cosine         # cosine = smooth + stable

# ─── Memory Savings ──────────────────────────────────────────────────────────
bf16: true
gradient_checkpointing: true
# Optional speed boosters:
# use_unsloth: true   (Colab-friendly, 2x faster)
# flash_attn: fa2     (requires FlashAttention2)
```

### Model → Template Mapping
```
meta-llama/Meta-Llama-3-8B-Instruct  → template: llama3
meta-llama/Llama-2-7b-chat-hf         → template: llama2
google/gemma-1.1-2b-it                → template: gemma
google/gemma-2-9b-it                  → template: gemma
mistralai/Mistral-7B-Instruct-v0.2    → template: mistral
Qwen/Qwen2-7B-Instruct                → template: qwen
microsoft/Phi-3-mini-4k-instruct      → template: phi
THUDM/chatglm3-6b                     → template: chatglm3
deepseek-ai/deepseek-llm-7b-chat      → template: deepseek
```

### VRAM Guide for Fine-tuning
```
GPU VRAM     →  Strategy
< 6 GB       →  QLoRA 4-bit + batch=1 + max_len=256 + unsloth
6-8 GB       →  QLoRA 4-bit + batch=1 + max_len=512
8-12 GB      →  QLoRA 4-bit + batch=1 + max_len=1024
16 GB        →  QLoRA 4-bit + batch=2 + max_len=2048
24 GB        →  LoRA FP16 + batch=2 or QLoRA + batch=4
40 GB        →  Full FT of 7B models + batch=4
80 GB (H100) →  Full FT of 13-30B models
```

### CLI One-liners
```bash
# Launch WebUI
llamafactory-cli webui

# Train with YAML
llamafactory-cli train train_gemma_qlora.yaml

# Export/merge LoRA into base model
llamafactory-cli export export_gemma.yaml

# Start OpenAI API server
llamafactory-cli api api_config.yaml

# Check version
llamafactory-cli version
```

In [ ]:
# ── Step 12.1: Full Pipeline Test ────────────────────────────────────────
import json, os

print("=" * 65)
print("        LLAMA FACTORY PIPELINE VERIFICATION")
print("=" * 65)

checks = {
    "LLaMA Factory cloned": os.path.exists("src"),
    "Requirements installed": True,  # If we got here, yes
    "Dataset (Alpaca)": os.path.exists("data/unix_alpaca.json"),
    "Dataset (ShareGPT)": os.path.exists("data/unix_sharegpt.json"),
    "Dataset (DPO)": os.path.exists("data/unix_dpo.json"),
    "Dataset (KTO)": os.path.exists("data/unix_kto.json"),
    "Dataset registered": os.path.exists("data/dataset_info.json"),
    "YAML: SFT Config": os.path.exists("train_gemma_qlora.yaml"),
    "YAML: LLaMA-3 Config": os.path.exists("train_llama3_qlora.yaml"),
    "YAML: DPO Config": os.path.exists("train_dpo.yaml"),
    "YAML: Export Config": os.path.exists("export_gemma.yaml"),
    "YAML: API Config": os.path.exists("api_config.yaml"),
    "YAML: Unsloth Config": os.path.exists("train_unsloth.yaml"),
    "YAML: GaLore Config": os.path.exists("train_galore.yaml"),
}

all_pass = True
for check, status in checks.items():
    icon = "✅" if status else "❌"
    if not status:
        all_pass = False
    print(f"  {icon} {check}")

print()
if all_pass:
    print("🎉 All checks passed! Ready to fine-tune.")
else:
    print("⚠️  Some checks failed. Run the respective cells above.")

print()
print("📌 Next Steps:")
print("  1. Run: llamafactory-cli webui  (GUI approach)")
print("  2. Run: llamafactory-cli train train_gemma_qlora.yaml  (CLI)")
print("  3. Run: llamafactory-cli export export_gemma.yaml  (after training)")
print("  4. Run: llamafactory-cli api api_config.yaml  (production serving)")